In [ ]:
from pyspark.sql import SparkSession
access_key = "key"

# Initialize Spark session
spark = SparkSession.builder \
    .appName("bronze-silver") \
    .config('spark.jars.packages', 'io.delta:delta-core_2.12:1.2.1,org.apache.hadoop:hadoop-azure:3.3.4,com.microsoft.azure:azure-storage:7.0.1,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0') \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.azure.account.key.jboutlook22.dfs.core.windows.net", access_key) \
    .config("spark.databricks.delta.formatCheck.enabled", "false") \
    .config("spark.streaming.stopGracefullyonShutdown", True) \
    .config("spark.sql.shuffle.partitions", 4) \
    .getOrCreate()

In [ ]:
spark

In [ ]:
# Bronze output base path in ADLS
bronze_path = "abfss://bronze@jboutlook22.dfs.core.windows.net/os_24hr_test"

In [ ]:
# Load delta table to be compacted
sparkdf = spark.read.format("delta") \
  .load(bronze_path)

In [ ]:
sparkdf.printSchema()
sparkdf.show()

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType

json_schema = StructType([
    StructField("payload", StructType([
        StructField("collection", StructType([
            StructField("slug", StringType(), True)
        ]), True),
        StructField("event_timestamp", StringType(), True),
        StructField("item", StructType([
            StructField("chain", StructType([
                StructField("name", StringType(), True)
            ]), True),
            StructField("metadata", StructType([
                StructField("image_url", StringType(), True),
                StructField("metadata_url", StringType(), True),
                StructField("name", StringType(), True)
            ]), True),
            StructField("nft_id", StringType(), True)
        ]), True),
        StructField("maker", StructType([
            StructField("address", StringType(), True)
        ]), True),
        StructField("order_hash", StringType(), True),
        StructField("payment_token", StructType([
            StructField("address", StringType(), True),
            StructField("decimals", LongType(), True),
            StructField("eth_price", StringType(), True),
            StructField("name", StringType(), True),
            StructField("symbol", StringType(), True),
            StructField("usd_price", StringType(), True)
        ]), True),
        StructField("protocol_address", StringType(), True),
        StructField("protocol_data", StructType([
            StructField("parameters", StructType([
                StructField("conduitKey", StringType(), True),
                StructField("consideration", ArrayType(StructType([
                    StructField("endAmount", StringType(), True),
                    StructField("identifierOrCriteria", StringType(), True),
                    StructField("itemType", LongType(), True),
                    StructField("recipient", StringType(), True),
                    StructField("startAmount", StringType(), True),
                    StructField("token", StringType(), True)
                ])), True),
                StructField("counter", StringType(), True),
                StructField("endTime", StringType(), True),
                StructField("offer", ArrayType(StructType([
                    StructField("endAmount", StringType(), True),
                    StructField("identifierOrCriteria", StringType(), True),
                    StructField("itemType", LongType(), True),
                    StructField("startAmount", StringType(), True),
                    StructField("token", StringType(), True)
                ])), True),
                StructField("offerer", StringType(), True),
                StructField("orderType", LongType(), True),
                StructField("salt", StringType(), True),
                StructField("startTime", StringType(), True),
                StructField("totalOriginalConsiderationItems", LongType(), True),
                StructField("zone", StringType(), True),
                StructField("zoneHash", StringType(), True)
            ]), True),
            StructField("signature", StringType(), True)
        ]), True),
        StructField("quantity", LongType(), True),
        StructField("sale_price", StringType(), True),
        StructField("taker", StructType([
            StructField("address", StringType(), True)
        ]), True),
        StructField("transaction", StructType([
            StructField("hash", StringType(), True),
            StructField("timestamp", StringType(), True)
        ]), True)
    ]), True),
    StructField("sent_at", StringType(), True)
])

In [ ]:
from pyspark.sql.functions import from_json,col

minimal_df = sparkdf.selectExpr(
    "payload.collection.slug as collection", \
    "payload.item.metadata.image_url", \
    "payload.item.metadata.name as nft_name", \
    "payload.item.nft_id", \
    "payload.maker.address as maker_address", \
    "payload.payment_token.decimals", \
    "payload.payment_token.eth_price", \
    "payload.payment_token.symbol", \
    "payload.payment_token.usd_price", \
    "payload.quantity", \
    "payload.sale_price", \
    "payload.taker.address as buyer_address", \
    "payload.transaction.hash as transaction_hash", \
    "payload.transaction.timestamp as transaction_timestamp",)

In [ ]:
minimal_df.printSchema()
minimal_df.show()